# 15 · The report agent: writing files through a stdio MCP server

Stage 3, step 5, the last agent. Say *"write a report of our conversation"* to the router and you get a Markdown file in `reports/`: a title and summary, every question with its answer, and the sources and links.

New in this step:
1. **A stdio MCP server.** The OKF and arXiv servers run in their own windows and we connect over HTTP. The official **filesystem server** is different: our agent **starts it as a child process** and talks to it through stdin/stdout. No window, no port.
2. **The router passes context.** Only the router has seen the whole conversation, so it sends it along with the message to the report agent.
3. **A tool that changes things.** Until now every tool only read data. `write_file` creates files on your disk, so it's locked to one folder by the **server itself**.

```
 "write a report of our conversation"
        │
   okf_router ── adds the conversation so far ──► A2A ──► report_agent
                                                           prepare ─┬─► summarize (qwen: title + bullets)
                                                                    │        │
                                                                    │      save ── write_file ──► filesystem MCP server ──► reports/xyz.md
                                                                    │                             (npx child process, stdio,
                                                                    ├─► list_reports              only allowed in reports/)
                                                                    └─► (nothing to report)
                                                           respond → log
```

## 0. Before you start

1. **Node.js** (once). The filesystem server is a Node.js program, run with `npx`. Check in a terminal: `node --version` and `npx --version`. If missing, install the **LTS** from https://nodejs.org (or `winget install OpenJS.NodeJS.LTS`), then **close and reopen** the terminal.
2. **Restart the backend**: close the 3 windows, run `start_backend.bat`. Still three windows. The filesystem server has no window: the agent server starts it the first time a report is written. The first time, `npx` downloads it, which takes about 10–30 s.

In [2]:
%load_ext autoreload
%autoreload 2

import os
import shutil
import socket
import subprocess
import time
from pathlib import Path

import pandas as pd
import psycopg

from okf_agent import AgentConfig
from okf_agent.a2a_client import A2AClient, langgraph_card_url

AGENT_URL = os.getenv("OKF_AGENT_URL", "http://127.0.0.1:2024")
config = AgentConfig.load()

npx = shutil.which("npx")
print("npx:          ", npx or "NOT FOUND  <- install Node.js LTS, reopen the terminal and VS Code")
if npx:
    print("node version: ", subprocess.run([shutil.which("node"), "--version"], capture_output=True, text=True).stdout.strip())
print("reports dir:  ", config.reports_dir)
print("fs server:    ", config.fs_mcp_connection())

def port_open(url):
    host, port = url.split("//")[1].split("/")[0].split(":")
    with socket.socket() as s:
        s.settimeout(2)
        return s.connect_ex((host, int(port))) == 0
print("agent server up:", port_open(AGENT_URL))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
npx:           NOT FOUND  <- install Node.js LTS, reopen the terminal and VS Code
reports dir:   c:\Users\Swapn\OneDrive\Documents\GitHub\OKF\reports
fs server:     {'command': 'npx', 'args': ['-y', '@modelcontextprotocol/server-filesystem@2026.8.31', 'c:\\Users\\Swapn\\OneDrive\\Documents\\GitHub\\OKF\\reports'], 'transport': 'stdio'}
agent server up: True


## 1. stdio vs HTTP: two ways to run an MCP server

| | HTTP (OKF server, arXiv server) | **stdio** (filesystem server) |
|---|---|---|
| who starts it | you, in its own window | **the client**, as a child process |
| how they talk | HTTP requests to a port | text over the child's stdin/stdout |
| lives | until you close the window | as long as the client keeps it |
| typical for | shared services, several clients | one client's private tool, e.g. desktop apps |

How the connection is described (`AgentConfig.fs_mcp_connection()`):

In [3]:
config.fs_mcp_connection()

{'command': 'npx',
 'args': ['-y',
  '@modelcontextprotocol/server-filesystem@2026.8.31',
  'c:\\Users\\Swapn\\OneDrive\\Documents\\GitHub\\OKF\\reports'],
 'transport': 'stdio'}

`"command": "npx", "args": ["-y", "@modelcontextprotocol/server-filesystem@2026.8.31", "<reports folder>"]`. That is: **start this program with these arguments**. The **last argument is the only folder the server will touch**. `@2026.8.31` pins the version. (npm's "latest" tag currently points to a brand-new `1.0.0`. Pinning means an update can't change things under us.)

### Keeping the child process alive: `PersistentMCP`

With `load_tools()` + `call_tool()` (what we use for HTTP), **every call would start a new child process**. With `npx` that's seconds per call; in my sandbox it was **71 s per call**, because npx checks the npm registry each time. So `okf_agent/mcp_tools.py` has `PersistentMCP`: start the process once, keep the session open in a background thread, and send every call to it.

```python
class PersistentMCP:
    def tools(self):
        if self._tools is None:                                # first use: start the child process
            self._loop = asyncio.ProactorEventLoop() if windows else asyncio.new_event_loop()
            threading.Thread(target=self._loop.run_forever, daemon=True).start()
            self._tools = run_coroutine_threadsafe(self._open(), self._loop).result()
        return self._tools

    async def _open(self):
        self._cm = MultiServerMCPClient({...}).session(self.name)    # starts npx + MCP handshake
        session = await self._cm.__aenter__()                         # ...and doesn't close it
        return {t.name: t for t in await load_mcp_tools(session)}

    def call(self, tool_name, args):
        return run_coroutine_threadsafe(self.tools()[tool_name].ainvoke(args), self._loop).result()
```

In my test: start 0.2 s (package already downloaded), then **3–10 ms per call** instead of a new process each time. On Windows the loop must be a `ProactorEventLoop`, because only that kind can start child processes.

Try it directly (this starts its **own** copy of the server, separate from the agent server's):

In [4]:
from okf_agent.mcp_tools import PersistentMCP

files = PersistentMCP("files", config.fs_mcp_connection())
t0 = time.perf_counter()
tools = files.tools()                       # first call: npx starts the server (downloads it the very first time)
print(f"{len(tools)} tools, started in {time.perf_counter() - t0:.1f} s\n")
for name, tool in tools.items():
    print(f"  {name:26s} {tool.description.splitlines()[0][:80]}")

UnsupportedOperation: fileno

## 2. The server enforces the folder

The agent's code only ever writes to `reports/`, but the real protection is the **server**: it refuses any path outside the folder it was started with, whatever the client asks. So even a bug, or a model that got a tool call wrong, can't write anywhere else.

In [ ]:
reports = Path(config.reports_dir)
files.call("create_directory", {"path": str(reports / "_tests")})     # keep test files out of the report list
inside = str(reports / "_tests" / "test.md")
outside = str(reports.parent / "_should_not_exist.md")

for path in (inside, outside):
    t0 = time.perf_counter()
    result = files.call("write_file", {"path": path, "content": "# test\n"})
    print(f"{path}\n   → {result}  ({(time.perf_counter() - t0) * 1000:.0f} ms)\n")

print("outside file exists?", Path(outside).exists())
print(files.call("list_directory", {"path": str(reports)}))

(The server has no delete tool at all: reading, writing, moving and listing only. That's another safety choice by its authors.)

In [ ]:
files.reset()        # stop this notebook's copy of the server (the child process ends)

## 3. The graph (`okf_agent/report.py`)

| node | does |
|---|---|
| `prepare` | splits the incoming message: the request, plus the conversation after `=== CONVERSATION ===`. Decides: **write** (there's a conversation), **list** ("list my reports"), or **empty** |
| `summarize` | **the only qwen call**: a title and 3–5 summary bullets, from a trimmed copy of the conversation |
| `save` | builds the Markdown **in code** and writes it with `write_file` |
| `list_reports` | `list_directory` on `reports/` |
| `respond` | the reply: the file path plus a preview |
| `log` | `qa_log`, `agent = report_agent`, `search_query` = the file path |

Why the answers are copied **word for word** and not rewritten by qwen: a report that misquotes "298/400" as "290/400" is worse than no report. qwen writes only what can't come from the conversation itself (title, summary). Everything else is assembled from what the agents actually said, including their citations and links.

The helpers are plain functions:

In [ ]:
from okf_agent.report import build_markdown, format_conversation, split_message, LIST_RE
from datetime import datetime

turns = [{"q": "What score did GPT-4 get on the bar exam?", "a": "GPT-4 scored 298/400 [topics/simulated-exams]."},
         {"q": "Find recent papers on MoE", "a": "- Mixtral [arXiv:2401.04088] (2024): ...\n\nLinks:\n- Mixtral of Experts: https://arxiv.org/abs/2401.04088"}]
message = "Write a report\n=== CONVERSATION ===\n" + format_conversation(turns)
print(message, "\n")
print("split back:", split_message(message)[0], "|", len(split_message(message)[1]), "turns\n")
print(build_markdown("Demo report", ["GPT-4 scored 298/400 on the bar exam."], turns, datetime.now()))

## 4. How the router sends the conversation

The router has the chat in its memory, the agents don't. So for agents in `NEEDS_CONVERSATION` (just `report_agent`), `route` attaches it:

```python
for agent in set(chosen) & NEEDS_CONVERSATION:
    turns = conversation_so_far(state["messages"][:-1])        # earlier turns, not this message
    parts[agent] = f"{question}\n{SEPARATOR}\n{format_conversation(turns)}" if turns else question
```

`conversation_so_far` skips greetings and earlier report requests, so they don't end up in the report. The router also knows the new agent only through its **card**: `report_agent` was added to `langgraph.json` and to the default `OKF_ROUTER_AGENTS`, plus two examples in the routing prompt ("Write a report of our conversation", "List my reports").

## 5. A real conversation, then a report

Everything goes through the router, like a user would. (Each answer takes your usual 10–20 s.)

In [ ]:
router = A2AClient.from_card(langgraph_card_url(AGENT_URL, "okf_router"))
conversation = {"id": None}

def chat(message, show=500):
    t0 = time.perf_counter()
    r = router.send(message, context_id=conversation["id"])
    conversation["id"] = r.context_id
    print(f"YOU: {message}   [{r.state}, {time.perf_counter() - t0:.1f} s]\n{'-' * 90}\n{r.text[:show]}{' …' if len(r.text) > show else ''}\n")


chat("Write a report of our conversation")          # nothing to report yet
chat("What score did GPT-4 get on the bar exam?")
chat("And GPT-3.5?")
chat("Find recent papers on mixture of experts language models")

In [ ]:
chat("Write a report of our conversation", show=1200)

In [ ]:
newest = max(Path(config.reports_dir).glob("*.md"), key=lambda p: p.stat().st_mtime)
print(newest, "\n")
print(newest.read_text(encoding="utf-8"))

Open the file in VS Code and press **Ctrl+Shift+V** for the formatted preview. Check that:
- the summary's numbers match the answers (qwen wrote the summary),
- the answers are word for word what the agents said,
- **Sources** lists the bundle concept ids and the arXiv links.

In [ ]:
chat("list my reports")

## 6. The log

In [ ]:
with psycopg.connect(config.postgres_url) as conn:
    cur = conn.execute("""SELECT id, agent, left(question, 50) AS question, search_query, refused, round(total_ms) AS ms
                          FROM qa_log ORDER BY id DESC LIMIT 12""")
    df = pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description])
df

## Where we are: stage 3 is done

| | built | MCP server | how it runs |
|---|---|---|---|
| `okf_agent` | answers from the bundle | **ours**: okf_mcp | HTTP :8000, own window |
| `paper_scout` | finds papers on arXiv | **existing**: arxiv-mcp-server | HTTP :8001, own window (uvx) |
| `report_agent` | writes reports | **existing**: official filesystem server | **stdio child process** (npx) |
| `okf_router` | the entry point | none, it calls the agents over **A2A** | |

All four are served by `langgraph dev` on :2024, log to `qa_log` in Postgres, and are traced in LangSmith.

**Next, stage 4: guardrails**, as graph nodes in the router:
- **input** (before `route`): too long or empty, prompt-injection patterns
- **output** (after `combine`): act on `bad_citations` (made-up bundle citations are still only logged today), and check the refusal wording